# 04 - Supabase Data Operations

Query the Supabase PostgreSQL database and list Supabase Storage buckets read-only. Every section uses the notebook's read-only database login from `DATABASE_URL`.

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

## 1. Query PostgreSQL

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT schemaname, tablename 
        FROM pg_tables 
        WHERE schemaname = 'public'
        LIMIT 10
    """))
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    display(df)

## 2. Supabase Storage

Atlas grants the Storage HTTP API only to the service-role key, and JupyterHub does not receive that key (see `services/supabase/README.md` §3.1). The anon key gets "permission denied" from Storage.

The notebook's read-only database login can still list bucket metadata from `storage.buckets`.

In [ ]:
# List storage buckets (read-only bucket metadata)
with engine.connect() as conn:
    buckets = conn.execute(text(
        "SELECT id, name, created_at FROM storage.buckets ORDER BY id"
    )).fetchall()
print("Storage Buckets:")
for bucket in buckets:
    print(f"  • {bucket.name} (created: {bucket.created_at})")

## 3. Database Version Info

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    version = result.fetchone()[0]
    print(f"PostgreSQL Version:\n{version}")